# 01 — LangChain Integration

This notebook covers your first interaction with an LLM via code — from a simple "hello world" call to structured outputs, system prompts, and conversational memory.

## Setup

Load environment variables and initialize the Gemini model.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)
print("Model ready!")

## Step 5 — First LLM Call & Model Parameters

### Understanding `llm.invoke()`

Let's make our first LLM call and examine what comes back.

The `invoke()` method returns an `AIMessage` object. The main fields:
- `.content` — the actual text response
- `.response_metadata` — dict with token counts, finish reason, etc.

The `temperature` parameter controls randomness:
- **0** = deterministic, always picks the most likely token
- **0.7** = creative, can produce more varied outputs


### Call invoke and inspect the output


In [ ]:
response = llm.invoke("What is the capital of France?")
print("--- Content ---")
print(response.content)
print("\n--- Response Metadata ---")
print(response.response_metadata)


### Compare Temperature Settings

Now call the same question at different temperature values and compare the style of the responses.


In [ ]:
llm_creative = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
)

print("=== Temperature = 0.7 ===")
response = llm_creative.invoke("What is a good way to learn Python?")
print(response.content)

print("\n=== Temperature = 0 (same question) === ")
response = llm.invoke("What is a good way to learn Python?")
print(response.content)


The temperature=0.7 response is usually more verbose and creative,
while temperature=0 stays concise and factual.


### System Prompts

**Understanding System vs Human roles:**
- `("system", ...)` — instructions for the model's behavior, persona, and constraints
- `("human", ...)` — the user's actual query or request

System messages set the rules; human messages are the user’s turn.

Let’s try a fun example: tell the model to act like a caveman.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a caveman. Use basic words only. Short and small sentences. No big talk."),
    ("human", "What is a computer?"),
])

messages = prompt.format_messages()
response = llm.invoke(messages)
print(response.content)


### Structured JSON Output

Now a practical use case: use the system prompt to enforce JSON output format. This is essential when you need machine-parseable responses.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Always respond in valid JSON."),
    ("human", "List 3 popular programming languages. Return as a JSON array of objects with keys: 'name', 'year_created', 'paradigm'."),
])

messages = prompt.format_messages()
response = llm.invoke(messages)
print(response.content)


## Step 6 — System Prompts & Out-of-Scope Refusal

System prompts define the persona and behavioral boundaries.

In [ ]:
system_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a strict programming and tech assistant. "
        "You ONLY answer questions related to programming, technology, and computer science. "
        "For any question outside this scope, politely refuse and explain your role."
    )),
    ("human", "{question}"),
])

chain = system_prompt | llm

print("=== IN-SCOPE ===")
print(chain.invoke({"question": "What is Python used for?"}).content)

print("\n=== OUT-OF-SCOPE ===")
print(chain.invoke({"question": "What is the best pizza topping?"}).content)

## Step 8 — Chat Memory

LLMs are **stateless** — they don't remember previous conversations. Every call is independent.

Let's demonstrate this problem, then fix it.

### Without Memory

Run this loop and try the sequence:
- Type: `My name is Budi`
- Type: `What is my name?`

The bot won't remember your name because no history is passed between turns.

In [ ]:
print("Chat started! Try: 'My name is Budi' then 'What is my name?'")
print("Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ("exit", "quit"):
        break

    prompt = ChatPromptTemplate.from_messages([("human", user_input)])
    response = (prompt | llm).invoke({})
    print(f"Bot: {response.content}\n")

### Manual Chat History

To make the LLM remember, we pass the full conversation history into each new prompt using a list of `("role", "content")` tuples.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [
    HumanMessage(content="What is the weather in Jakarta today?"),
    AIMessage(content="It is 32°C and partly cloudy in Jakarta."),
    HumanMessage(content="Should I bring an umbrella?"),
    AIMessage(content="There is a 30% chance of rain, so it might be safe without one, but a small umbrella wouldn't hurt."),
]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly weather assistant."),
    *chat_history,
    ("human", "What about tomorrow?"),
])

response = (prompt | llm).invoke({})
print(response.content)

### With Memory

Now the same name-remembering test, but with history accumulated between turns.

In [ ]:
print("Chat started! Try: 'My name is Budi' then 'What is my name?'")
print("Type 'exit' to quit.\n")

chat_history = []

while True:
    user_input = input("You: ")
    if user_input.lower() in ("exit", "quit"):
        break

    prompt = ChatPromptTemplate.from_messages(
        chat_history + [("human", user_input)]
    )
    response = (prompt | llm).invoke({})
    print(f"Bot: {response.content}\n")

    chat_history.append(("human", user_input))
    chat_history.append(("ai", response.content))